# RTLCoder-Deepseek-v1.1 on Colab

Runs the **same 18 generations** as the Llama / RTLCoder-GGUF pilot:
3 circuits × 2 prompts × 3 attempts.

**Model:** `ishorn5/RTLCoder-Deepseek-v1.1` (best official RTLCoder, GPU).
This is **not** the 4-bit Mistral GGUF already run on `ecs05`.

## Setup
1. Runtime → Change runtime type → **T4 GPU** (A100/L4 also fine).
2. Run all cells.
3. Download `rtlcoder-deepseek-v1.1.zip`.
4. On `ecs05`:

```tcsh
cd /home/ft2335/dataset
unzip -o rtlcoder-deepseek-v1.1.zip
python3 scripts/evaluate_generated.py --model-dir experiments/rtlcoder-deepseek-v1.1
```

Do not edit generated RTL. Do not overwrite `experiments/rtlcoder-v1.1-gguf-4bit/`.

In [ ]:
!nvidia-smi -L
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime → Change runtime type → T4 GPU'

In [ ]:
%pip install -q 'transformers>=4.40' accelerate sentencepiece
# Optional fallback if fp16 OOM on a small GPU:
%pip install -q bitsandbytes

In [ ]:
from pathlib import Path

ROOT = Path('/content/cdc_pilot')
PROMPT_DIR = ROOT / 'experiments' / 'prompts'
OUT_ROOT = ROOT / 'experiments' / 'rtlcoder-deepseek-v1.1'
PROMPT_DIR.mkdir(parents=True, exist_ok=True)

PROMPTS = {
'cdc_2phase.functional.md': r'''Generate synthesizable SystemVerilog for the following module:

module cdc_2phase #(
  parameter WIDTH = 1
)(
  input                  src_rst_ni,
  input                  src_clk_i,
  input      [WIDTH-1:0] src_data_i,
  input                  src_valid_i,
  output                 src_ready_o,

  input                  dst_rst_ni,
  input                  dst_clk_i,
  output     [WIDTH-1:0] dst_data_o,
  output                 dst_valid_o,
  input                  dst_ready_i
);

src_clk_i and dst_clk_i are independent asynchronous clocks with no fixed
frequency or phase relationship.

A source transfer is accepted on a rising src_clk_i edge when src_valid_i and
src_ready_o are both high. Every accepted source item must appear exactly once
at the destination, in the original order.

A destination transfer completes on a rising dst_clk_i edge when dst_valid_o
and dst_ready_i are both high. While dst_valid_o is high and dst_ready_i is
low, dst_valid_o must remain asserted and dst_data_o must remain stable.

The design may support one outstanding item. src_ready_o must be low whenever
a new source item cannot safely be accepted.

src_rst_ni and dst_rst_ni are active-low resets. Reset must return both
interfaces to an idle state and must not create a spurious destination
transaction.

Place the complete implementation, including any helper modules, in one source
file. Do not include a testbench, explanation, markdown, vendor primitives, or
the reference implementation.
''',
'cdc_2phase.cdc_explicit.md': r'''Generate synthesizable SystemVerilog for the following module:

module cdc_2phase #(
  parameter WIDTH = 1
)(
  input                  src_rst_ni,
  input                  src_clk_i,
  input      [WIDTH-1:0] src_data_i,
  input                  src_valid_i,
  output                 src_ready_o,

  input                  dst_rst_ni,
  input                  dst_clk_i,
  output     [WIDTH-1:0] dst_data_o,
  output                 dst_valid_o,
  input                  dst_ready_i
);

src_clk_i and dst_clk_i are independent asynchronous clocks with no fixed
frequency or phase relationship.

A source transfer is accepted on a rising src_clk_i edge when src_valid_i and
src_ready_o are both high. Every accepted source item must appear exactly once
at the destination, in the original order.

A destination transfer completes on a rising dst_clk_i edge when dst_valid_o
and dst_ready_i are both high. While dst_valid_o is high and dst_ready_i is
low, dst_valid_o must remain asserted and dst_data_o must remain stable.

The design may support one outstanding item. src_ready_o must be low whenever
a new source item cannot safely be accepted.

src_rst_ni and dst_rst_ni are active-low resets. Reset must return both
interfaces to an idle state and must not create a spurious destination
transaction.

The implementation must be safe for clock-domain and reset-domain crossings
and must pass structural CDC/RDC analysis with zero unsafe crossings. Reset
release must be safe in each clock domain, and transferred multi-bit data must
remain coherent. Select the architecture yourself.

Place the complete implementation, including any helper modules, in one source
file. Do not include a testbench, explanation, markdown, vendor primitives, or
the reference implementation.
''',
'async_fifo.functional.md': r'''Generate synthesizable Verilog-2001 for this module:

module async_fifo #(
  parameter DSIZE = 8,
  parameter ASIZE = 4,
  parameter FALLTHROUGH = "TRUE"
)(
  input  wire             wclk,
  input  wire             wrst_n,
  input  wire             winc,
  input  wire [DSIZE-1:0] wdata,
  output wire             wfull,
  output wire             awfull,

  input  wire             rclk,
  input  wire             rrst_n,
  input  wire             rinc,
  output wire [DSIZE-1:0] rdata,
  output wire             rempty,
  output wire             arempty
);

Implement a FIFO containing 2**ASIZE entries of DSIZE bits. wclk and rclk are
independent asynchronous clocks.

A write is accepted on a rising wclk edge when winc is high and wfull is low.
Writes attempted while full must not alter FIFO contents.

A read is accepted on a rising rclk edge when rinc is high and rempty is low.
Reads attempted while empty must not advance the FIFO.

Accepted data must be returned exactly once and in write order. wfull is
generated in the write domain and rempty in the read domain. awfull indicates
that the FIFO is approaching full, and arempty indicates that it is approaching
empty.

After reset, wfull must be low and rempty must be high. wrst_n and rrst_n are
active-low resets belonging to their respective clock domains.

When FALLTHROUGH equals "TRUE", rdata presents the current oldest unread word
without requiring an additional registered-read cycle. Otherwise, rdata may be
updated by an accepted read.

Place the complete implementation and helper modules in one file. Do not
include a testbench, explanation, markdown, vendor primitives, or reference
code.
''',
'async_fifo.cdc_explicit.md': r'''Generate synthesizable Verilog-2001 for this module:

module async_fifo #(
  parameter DSIZE = 8,
  parameter ASIZE = 4,
  parameter FALLTHROUGH = "TRUE"
)(
  input  wire             wclk,
  input  wire             wrst_n,
  input  wire             winc,
  input  wire [DSIZE-1:0] wdata,
  output wire             wfull,
  output wire             awfull,

  input  wire             rclk,
  input  wire             rrst_n,
  input  wire             rinc,
  output wire [DSIZE-1:0] rdata,
  output wire             rempty,
  output wire             arempty
);

Implement a FIFO containing 2**ASIZE entries of DSIZE bits. wclk and rclk are
independent asynchronous clocks.

A write is accepted on a rising wclk edge when winc is high and wfull is low.
Writes attempted while full must not alter FIFO contents.

A read is accepted on a rising rclk edge when rinc is high and rempty is low.
Reads attempted while empty must not advance the FIFO.

Accepted data must be returned exactly once and in write order. wfull is
generated in the write domain and rempty in the read domain. awfull indicates
that the FIFO is approaching full, and arempty indicates that it is approaching
empty.

After reset, wfull must be low and rempty must be high. wrst_n and rrst_n are
active-low resets belonging to their respective clock domains.

When FALLTHROUGH equals "TRUE", rdata presents the current oldest unread word
without requiring an additional registered-read cycle. Otherwise, rdata may be
updated by an accepted read.

The implementation must be safe for clock-domain and reset-domain crossings
and must pass structural CDC/RDC analysis with zero unsafe crossings. Reset
release must be safe in each clock domain, and transferred multi-bit data must
remain coherent. Select the architecture yourself.

Place the complete implementation and helper modules in one file. Do not
include a testbench, explanation, markdown, vendor primitives, or reference
code.
''',
'apbxclk.functional.md': r'''Generate synthesizable Verilog-2001 implementing an APB clock-domain bridge:

module apbxclk #(
  parameter C_APB_ADDR_WIDTH = 12,
  parameter C_APB_DATA_WIDTH = 32,
  parameter [0:0] OPT_REGISTERED = 1'b0
)(
  input  wire                         S_APB_PCLK,
  input  wire                         S_PRESETn,
  input  wire                         S_APB_PSEL,
  input  wire                         S_APB_PENABLE,
  output reg                          S_APB_PREADY,
  input  wire [C_APB_ADDR_WIDTH-1:0]  S_APB_PADDR,
  input  wire                         S_APB_PWRITE,
  input  wire [C_APB_DATA_WIDTH-1:0]  S_APB_PWDATA,
  input  wire [C_APB_DATA_WIDTH/8-1:0] S_APB_PWSTRB,
  input  wire [2:0]                   S_APB_PPROT,
  output wire [C_APB_DATA_WIDTH-1:0]  S_APB_PRDATA,
  output wire                         S_APB_PSLVERR,

  input  wire                         M_APB_PCLK,
  output reg                          M_PRESETn,
  output reg                          M_APB_PSEL,
  output reg                          M_APB_PENABLE,
  input  wire                         M_APB_PREADY,
  output wire [C_APB_ADDR_WIDTH-1:0]  M_APB_PADDR,
  output wire                         M_APB_PWRITE,
  output wire [C_APB_DATA_WIDTH-1:0]  M_APB_PWDATA,
  output wire [C_APB_DATA_WIDTH/8-1:0] M_APB_PWSTRB,
  output wire [2:0]                   M_APB_PPROT,
  input  wire [C_APB_DATA_WIDTH-1:0]  M_APB_PRDATA,
  input  wire                         M_APB_PSLVERR
);

S_APB_PCLK and M_APB_PCLK are independent asynchronous clocks.

Accept standard APB transfers on the S_APB interface. Forward each accepted
request exactly once to the M_APB interface using a normal APB setup phase
followed by an access phase. Keep the downstream request fields stable until
M_APB_PREADY completes the transfer.

Return read data and slave-error status to the source interface. Assert
S_APB_PREADY only when the corresponding downstream transaction has completed.
Do not lose, duplicate, or reorder requests. Supporting one outstanding
transaction is sufficient.

S_PRESETn is the active-low source reset. M_PRESETn is an active-low reset
output for the destination domain. Both interfaces must remain inactive during
reset, and reset must not create a transaction.

OPT_REGISTERED selects whether crossing payload and response fields are
explicitly registered. Both parameter settings must preserve APB behavior.

Return one complete source file only. Do not include a testbench, explanation,
markdown, vendor primitives, or reference code.
''',
'apbxclk.cdc_explicit.md': r'''Generate synthesizable Verilog-2001 implementing an APB clock-domain bridge:

module apbxclk #(
  parameter C_APB_ADDR_WIDTH = 12,
  parameter C_APB_DATA_WIDTH = 32,
  parameter [0:0] OPT_REGISTERED = 1'b0
)(
  input  wire                         S_APB_PCLK,
  input  wire                         S_PRESETn,
  input  wire                         S_APB_PSEL,
  input  wire                         S_APB_PENABLE,
  output reg                          S_APB_PREADY,
  input  wire [C_APB_ADDR_WIDTH-1:0]  S_APB_PADDR,
  input  wire                         S_APB_PWRITE,
  input  wire [C_APB_DATA_WIDTH-1:0]  S_APB_PWDATA,
  input  wire [C_APB_DATA_WIDTH/8-1:0] S_APB_PWSTRB,
  input  wire [2:0]                   S_APB_PPROT,
  output wire [C_APB_DATA_WIDTH-1:0]  S_APB_PRDATA,
  output wire                         S_APB_PSLVERR,

  input  wire                         M_APB_PCLK,
  output reg                          M_PRESETn,
  output reg                          M_APB_PSEL,
  output reg                          M_APB_PENABLE,
  input  wire                         M_APB_PREADY,
  output wire [C_APB_ADDR_WIDTH-1:0]  M_APB_PADDR,
  output wire                         M_APB_PWRITE,
  output wire [C_APB_DATA_WIDTH-1:0]  M_APB_PWDATA,
  output wire [C_APB_DATA_WIDTH/8-1:0] M_APB_PWSTRB,
  output wire [2:0]                   M_APB_PPROT,
  input  wire [C_APB_DATA_WIDTH-1:0]  M_APB_PRDATA,
  input  wire                         M_APB_PSLVERR
);

S_APB_PCLK and M_APB_PCLK are independent asynchronous clocks.

Accept standard APB transfers on the S_APB interface. Forward each accepted
request exactly once to the M_APB interface using a normal APB setup phase
followed by an access phase. Keep the downstream request fields stable until
M_APB_PREADY completes the transfer.

Return read data and slave-error status to the source interface. Assert
S_APB_PREADY only when the corresponding downstream transaction has completed.
Do not lose, duplicate, or reorder requests. Supporting one outstanding
transaction is sufficient.

S_PRESETn is the active-low source reset. M_PRESETn is an active-low reset
output for the destination domain. Both interfaces must remain inactive during
reset, and reset must not create a transaction.

OPT_REGISTERED selects whether crossing payload and response fields are
explicitly registered. Both parameter settings must preserve APB behavior.

The implementation must be safe for clock-domain and reset-domain crossings
and must pass structural CDC/RDC analysis with zero unsafe crossings. Reset
release must be safe in each clock domain, and transferred multi-bit data must
remain coherent. Select the architecture yourself.

Return one complete source file only. Do not include a testbench, explanation,
markdown, vendor primitives, or reference code.
''',
}

for name, text in PROMPTS.items():
    (PROMPT_DIR / name).write_text(text)
print('wrote', len(PROMPTS), 'prompts to', PROMPT_DIR)

In [ ]:
import time, json, re, traceback
from datetime import datetime, timezone
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'ishorn5/RTLCoder-Deepseek-v1.1'
TEMPERATURE = 0.2
TOP_P = 0.95
MAX_NEW_TOKENS = 2048
CIRCUITS = ['cdc_2phase', 'async_fifo', 'apbxclk']
PROMPT_TYPES = ['functional', 'cdc_explicit']

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
load_dtype = 'fp16'
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map=0, trust_remote_code=True
    )
except Exception as exc:
    print('fp16 load failed:', exc)
    print('retrying 8-bit...')
    load_dtype = 'int8'
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, load_in_8bit=True, device_map=0, trust_remote_code=True
    )
model.eval()
print('loaded', MODEL_ID, load_dtype)

In [ ]:
def extract_verilog(text: str) -> str:
    """Official RTLCoder-Deepseek post-process from their README."""
    blocks = re.findall(r'```(?:systemverilog|verilog|sv|v)?\s*(.*?)```', text, flags=re.S)
    s_full = max(blocks, key=len) if blocks else text
    if len(s_full.split('endmodulemodule', 1)) == 2:
        s = s_full.split('endmodulemodule', 1)[0] + '\nendmodule'
    elif 'endmodule' in s_full:
        s = s_full.rsplit('endmodule', 1)[0] + '\nendmodule'
    else:
        s = s_full.strip()
    if s.find('top_module') != -1:
        s = s.split('top_module', 1)[0]
        if 'endmodule' in s:
            s = s.rsplit('endmodule', 1)[0] + '\nendmodule'
    index = s.rfind('tb_module')
    if index == -1:
        index = s.find('testbench')
    if index != -1:
        s_tmp = s[:index]
        if 'endmodule' in s_tmp:
            s = s_tmp.rsplit('endmodule', 1)[0] + '\nendmodule'
    return s.strip() + '\n'


def next_attempt(base: Path) -> Path:
    existing = [
        int(p.name.split('-')[1])
        for p in base.glob('attempt-*')
        if p.name.split('-')[-1].isdigit()
    ]
    return base / f'attempt-{max(existing, default=0) + 1:03d}'


def generate_one(circuit: str, prompt_type: str):
    prompt_path = PROMPT_DIR / f'{circuit}.{prompt_type}.md'
    prompt = prompt_path.read_text()
    out_dir = next_attempt(OUT_ROOT / circuit / prompt_type)
    out_dir.mkdir(parents=True, exist_ok=False)
    (out_dir / 'generated').mkdir()
    (out_dir / 'prompt.md').write_text(prompt)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    t0 = time.time()
    with torch.no_grad():
        sample = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
        )
    elapsed = time.time() - t0
    text = tokenizer.decode(sample[0], skip_special_tokens=False)
    (out_dir / 'response.txt').write_text(text if text.endswith('\n') else text + '\n')
    (out_dir / 'generated' / f'{circuit}.v').write_text(extract_verilog(text))
    meta = {
        'provider': 'colab-gpu',
        'model': MODEL_ID,
        'load_dtype': load_dtype,
        'circuit': circuit,
        'prompt_type': prompt_type,
        'temperature': TEMPERATURE,
        'top_p': TOP_P,
        'max_new_tokens': MAX_NEW_TOKENS,
        'elapsed_s': round(elapsed, 3),
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'gpu': torch.cuda.get_device_name(0),
        'note': 'Official RTLCoder-Deepseek-v1.1. Same prompts as Llama / GGUF pilots.',
    }
    (out_dir / 'metadata.json').write_text(json.dumps(meta, indent=2) + '\n')
    print(f'{circuit}/{prompt_type}/{out_dir.name}  {elapsed:.1f}s')
    return out_dir

jobs = []
for circuit in CIRCUITS:
    for prompt_type in PROMPT_TYPES:
        have = len(list((OUT_ROOT / circuit / prompt_type).glob('attempt-*')))
        jobs.extend([(circuit, prompt_type)] * max(0, 3 - have))
print('jobs remaining', len(jobs))
for circuit, prompt_type in jobs:
    generate_one(circuit, prompt_type)
print('done')

In [ ]:
from google.colab import files
import shutil
n = len(list(OUT_ROOT.glob('*/*/attempt-*/generated/*.v')))
print('generated files', n)
zip_path = shutil.make_archive('/content/rtlcoder-deepseek-v1.1', 'zip', ROOT, 'experiments/rtlcoder-deepseek-v1.1')
print(zip_path)
files.download(zip_path)